In [ ]:
#RAG

45

In [2]:
import os
from getpass import getpass

from dotenv import load_dotenv
load_dotenv()

MESHAPI_BASE_URL = os.getenv("MESHAPI_BASE_URL", "https://api.meshapi.ai")
MESHAPI_TOKEN = os.getenv("MESH_API_KEY") or os.getenv("MESHAPI_TOKEN") or getpass("MeshAPI token (rsk_...): ")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY") or getpass("Pinecone API key: ")


In [5]:
PINECONE_INDEX_NAME = "meshapi-demo-kb"
PINECONE_CLOUD = "aws"
PINECONE_REGION = "us-east-1"
EMBEDDING_DIMENSIONS = 1024

In [6]:
# OPEN THE MESHAPI CLIENT

from meshapi import MeshAPI

# Native MeshAPI SDK client -- used for chat, embeddings, and model discovery
client = MeshAPI(base_url=MESHAPI_BASE_URL, token=MESHAPI_TOKEN)
print("MeshAPI client ready.")


MeshAPI client ready.


In [ ]:
#MODELS



In [16]:
FAST_MODEL = "openai/gpt-4o-mini"
SMART_MODEL = "mistral/mistral-large-3-675b-instruct"
EMBEDDING_MODEL = "openai/text-embedding-3-small"

EMBEDDING_DIMENSIONS = 1024

print("Models configured successfully.")


Models configured successfully.


In [7]:
# ask() helper -- one chat completion, any model

from meshapi import ChatCompletionParams, ChatMessage

def ask(model, prompt, temperature=0.4, max_tokens=350):
    resp = client.chat.completions.create(
        ChatCompletionParams(
            model=model,
            messages=[ChatMessage(role="user", content=prompt)],
            temperature=temperature,
            max_tokens=max_tokens,
        )
    )
    return resp.choices[0].message.content


In [ ]:
#EMBEDDINGS


In [8]:
from meshapi import EmbeddingsParams

def mesh_embed(texts):
    resp = client.embeddings.create(
        EmbeddingsParams(model=EMBEDDING_MODEL, input=texts, dimensions=EMBEDDING_DIMENSIONS)
    )
    return [d.embedding for d in sorted(resp.data, key=lambda d: d.index)]


In [11]:
import time

from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=PINECONE_API_KEY)

if PINECONE_INDEX_NAME not in [idx["name"] for idx in pc.list_indexes()]:
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=EMBEDDING_DIMENSIONS,
        metric="cosine",
        spec=ServerlessSpec(cloud=PINECONE_CLOUD, region=PINECONE_REGION),
    )
    while not pc.describe_index(PINECONE_INDEX_NAME).status["ready"]:
        time.sleep(1)

index = pc.Index(PINECONE_INDEX_NAME)
print(index.describe_index_stats())


DescribeIndexStatsResponse(dimension=1024, total_vector_count=0, metric='cosine', namespaces=0)


In [12]:
#Added Knowldege databse

# A small fictional support-docs corpus for "Nimbus Cloud" storage. Swap this for your own docs later -- the pipeline below doesn't care where the text came from.

knowledge_base = [
    {"id": "doc-1", "title": "Refund Policy", "text": "Nimbus Cloud offers a 30-day money-back guarantee on all annual plans. Monthly plans can be cancelled anytime but are not eligible for partial refunds. Refund requests must be submitted through the billing portal within the eligibility window."},
    {"id": "doc-2", "title": "Storage Limits", "text": "The Starter plan includes 100GB of storage, Pro includes 2TB, and Enterprise is negotiated per contract. Exceeding your plan's limit pauses new uploads until you upgrade or free up space; existing files remain accessible."},
    {"id": "doc-3", "title": "Data Retention", "text": "Deleted files move to a Trash folder and are permanently removed after 30 days. Account cancellation triggers a 90-day data retention window before permanent deletion, during which reactivation restores all data."},
    {"id": "doc-4", "title": "Sharing & Permissions", "text": "Files can be shared via link (view or edit access) or invited by email with role-based permissions: Viewer, Commenter, Editor, Owner. Shared links can be password-protected and set to expire after a chosen number of days."},
    {"id": "doc-5", "title": "Two-Factor Authentication", "text": "2FA is optional for Starter and Pro plans but mandatory for all Enterprise accounts. Supported methods are authenticator apps (TOTP) and SMS. Recovery codes are generated once and shown only at setup time."},
    {"id": "doc-6", "title": "API Rate Limits", "text": "The Nimbus Cloud API allows 100 requests per minute on Starter, 1000 on Pro, and custom limits on Enterprise. Exceeding the limit returns HTTP 429 with a Retry-After header indicating when to resume."},
    {"id": "doc-7", "title": "Plan Downgrades", "text": "Downgrading takes effect at the end of the current billing cycle. If your stored data exceeds the new plan's limit, you'll have a 14-day grace period to remove files before uploads are paused."},
    {"id": "doc-8", "title": "Support Response Times", "text": "Starter plan support responds within 48 hours via email. Pro plan support responds within 24 hours and includes live chat. Enterprise customers get a dedicated support contact with a 4-hour SLA."},
]


In [13]:
# Chunk the knowledge base

def chunk_text(text, max_chars=500, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + max_chars
        chunks.append(text[start:end])
        start = end - overlap
    return chunks

chunks = []
for doc in knowledge_base:
    for i, c in enumerate(chunk_text(doc["text"])):
        chunks.append({"doc_id": doc["id"], "title": doc["title"], "chunk_index": i, "text": c})

print(f"{len(chunks)} chunks from {len(knowledge_base)} documents.")

8 chunks from 8 documents.


In [17]:
# Embed + upsert into Pinecone
# EMBEDD -> VECTOR DB
embeddings = mesh_embed([c["text"] for c in chunks])

pinecone_vectors = [
    {
        "id": f"{c['doc_id']}-{c['chunk_index']}",
        "values": emb,
        "metadata": {"title": c["title"], "text": c["text"], "doc_id": c["doc_id"]},
    }
    for c, emb in zip(chunks, embeddings)
]

index.upsert(vectors=pinecone_vectors)
print(f"Upserted {len(pinecone_vectors)} chunks into Pinecone index '{PINECONE_INDEX_NAME}'.")


Upserted 8 chunks into Pinecone index 'meshapi-demo-kb'.


In [ ]:
# DATA INGESTION
# DATA - CHUNKS - EMBEDDINGS - VEC DB
#RETRIVE?

In [ ]:
# VECTOR SEARCH
#RETERVIAL DATA
def retrieve(query, top_k=3):
    query_embedding = mesh_embed([query])[0]
    results = index.query(vector=query_embedding, top_k=top_k, include_metadata=True)
    return [
        {"score": m["score"], "title": m["metadata"]["title"], "text": m["metadata"]["text"]}
        for m in results["matches"]
    ]

for r in retrieve("How much storage do I get on the Pro plan?"):
    print(f"[{r['score']:.3f}] {r['title']}: {r['text'][:100]}...")


[0.687] Storage Limits: The Starter plan includes 100GB of storage, Pro includes 2TB, and Enterprise is negotiated per contr...
[0.455] Support Response Times: Starter plan support responds within 48 hours via email. Pro plan support responds within 24 hours a...
[0.365] Plan Downgrades: Downgrading takes effect at the end of the current billing cycle. If your stored data exceeds the ne...


In [19]:
# PLAIN RAG
# The simplest useful thing: retrieve context, stuff it in a prompt, ask a fast model directly via the native MeshAPI SDK (no framework needed for a one-shot call like this).

def rag_answer(question, model=FAST_MODEL, top_k=3):
    hits = retrieve(question, top_k=top_k)
    context = "\n\n".join(f"[{h['title']}] {h['text']}" for h in hits)
    prompt = f"""Answer the question using ONLY the context below. If the answer isn't in the context, say you don't know.

Context:
{context}

Question: {question}"""
    return ask(model, prompt, temperature=0.2, max_tokens=300), hits

answer, sources = rag_answer("What happens if I go over my storage limit?")
print(answer)
print("\nSources:", [s["title"] for s in sources])


If you exceed your plan's storage limit, new uploads will be paused until you upgrade or free up space, but existing files will remain accessible.

Sources: ['Storage Limits', 'Plan Downgrades', 'Data Retention']


In [ ]:
# #MULTI AGENT SYSTEM :-
# (1) Research agent
# (2) Writer Agent
# (3) Critic Agent(what write has wrote)

#INDEPENDENT OF LANGCHAIN

# CHATOPEN - USING LANGCHAIN -- STANDARAD OPENAI COMPATIBLE - hit client


In [29]:
# MeshAPI is an OpenAI-compatible gateway, so langchain_openai.ChatOpenAI can talk to it directly -- just point base_url at MeshAPI and pass your rsk_... token as api_key. This is the documented, recommended way to plug any OpenAI-compatible gateway/proxy into LangChain.

from langchain_openai import ChatOpenAI

MESHAPI_OPENAI_BASE_URL = f"{MESHAPI_BASE_URL}/v1"

fast_chat = ChatOpenAI(base_url=MESHAPI_OPENAI_BASE_URL, api_key=MESHAPI_TOKEN, model=FAST_MODEL, temperature=0.3)
smart_chat = ChatOpenAI(base_url=MESHAPI_OPENAI_BASE_URL, api_key=MESHAPI_TOKEN, model=SMART_MODEL, temperature=0.4)

print("LangChain chat models wired to MeshAPI:", FAST_MODEL, "|", SMART_MODEL)


LangChain chat models wired to MeshAPI: openai/gpt-4o-mini | mistral/mistral-large-3-675b-instruct


In [30]:
#TOOL - SEARCH KNOWLDEGE BASE 

import json

from langchain_core.tools import tool

@tool
def search_knowledge_base(query: str) -> str:
    """Search the Nimbus Cloud knowledge base for relevant policy/product information.

    Args:
        query: the search query text
    """
    hits = retrieve(query, top_k=3)
    return json.dumps(hits)


In [35]:
# Researcher agent (FAST_MODEL, via create_agent with a tool)

from langchain.agents import create_agent

researcher = create_agent(
    model=fast_chat,
    tools=[search_knowledge_base],
    system_prompt="""You are a research agent.

Your job is to research the user's question using the
Nimbus Cloud knowledge base.

Rules:
- Use the search_knowledge_base tool to find relevant information.
- Do not invent information.
- Use only information found in the knowledge base.
- Return concise research notes for the Writer agent.
- Include all important facts relevant to the user's question.
"""
)

In [34]:
# Writer agent (SMART_MODEL, via create_agent with no tools)



from langchain.agents import create_agent

writer = create_agent(
    model=smart_chat,
    tools=[],
    system_prompt="""You are a professional answer-writing agent.

Your job is to write a clear, accurate, and helpful answer
to the user's question using the research notes provided
by the Researcher.

Rules:
- Use only the information provided in the research notes.
- Do not invent or assume facts.
- Be concise and easy to understand.
- Directly answer the user's question.
- If the research notes do not contain enough information,
  clearly say that the information is insufficient.
"""
)

def run_writer(question, research_notes):
    result = writer.invoke({
        "messages": [
            {
                "role": "user",
                "content": f"""User question:
{question}

Research notes:
{research_notes}

Write the final answer for the user using the research notes."""
            }
        ]
    })

    return result["messages"][-1].content

In [33]:
import langchain
import langchain_openai

print("LangChain:", langchain.__version__)
print("LangChain OpenAI imported successfully!")

LangChain: 1.4.0
LangChain OpenAI imported successfully!


In [36]:
#Cretic agent - strctured critics (requirred pydantic)
#Criic agent is qualtity checkinh agent 
from pydantic import BaseModel, Field

class Critique(BaseModel):
    verdict: str = Field(description="'pass' or 'revise'")
    reason: str
    missing_info: str = Field(default="", description="what's missing, if verdict is 'revise'")

critic_agent = create_agent(
    model=smart_chat,
    system_prompt=(
        "You are a quality reviewer for customer support answers. Grade the draft strictly: does it "
        "fully and accurately answer the question using only reasonable support-agent knowledge?"
    ),
    response_format=Critique,
)

def run_critic(question, draft):
    prompt = f"Question: {question}\n\nDraft answer: {draft}\n\nGrade this draft."
    result = critic_agent.invoke({"messages": [{"role": "user", "content": prompt}]})
    return result["structured_response"]


In [44]:
def run_researcher(question):
    result = researcher.invoke({
        "messages": [
            {
                "role": "user",
                "content": question
            }
        ]
    })

    return result["messages"][-1].content

In [47]:
# Orchestrate the full pipeline

# Orchestrate the full pipeline

def research_write_review(question):
    print(f"Question: {question}\n")

    # Researcher
    notes = run_researcher(question)
    print(f"--- Researcher ({FAST_MODEL}) ---")
    print(notes)
    print()

    # Writer
    draft = run_writer(question, notes)
    print(f"--- Writer ({SMART_MODEL}) ---")
    print(draft)
    print()

    # Critic
    critique = run_critic(question, draft)
    print(f"--- Critic ({SMART_MODEL}) ---")
    print(f"Verdict: {critique.verdict}")
    print(f"Reason: {critique.reason}")
    print()

    return draft

In [48]:
final = research_write_review(
    "If I cancel my monthly plan halfway through, do I get money back?"
)

print("\n=== FINAL ANSWER ===")
print(final)

Question: If I cancel my monthly plan halfway through, do I get money back?

--- Researcher (openai/gpt-4o-mini) ---
If you cancel your monthly plan halfway through, you will not receive any money back. Monthly plans can be canceled at any time, but they are not eligible for partial refunds.

--- Writer (mistral/mistral-large-3-675b-instruct) ---
No, you will not get any money back if you cancel your monthly plan halfway through. Monthly plans can be canceled at any time, but they are not eligible for partial refunds.

--- Critic (mistral/mistral-large-3-675b-instruct) ---
Verdict: revise
Reason: The draft answer is **mostly accurate** but lacks clarity on whether the cancellation takes effect immediately or if the user retains access for the remainder of the paid period. While it correctly states that no partial refund will be issued, it fails to address whether the user can continue using the service until the end of the billing cycle. This is a common point of confusion for customer

In [49]:
research_write_review("Is two-factor authentication required on the Pro plan?")


Question: Is two-factor authentication required on the Pro plan?

--- Researcher (openai/gpt-4o-mini) ---
Two-factor authentication (2FA) is optional for the Pro plan. It is mandatory only for Enterprise accounts. Supported methods for 2FA include authenticator apps (TOTP) and SMS.

--- Writer (mistral/mistral-large-3-675b-instruct) ---
No, two-factor authentication (2FA) is **not required** on the Pro plan—it is optional. 2FA is only mandatory for Enterprise accounts. Supported methods include authenticator apps (TOTP) and SMS.

--- Critic (mistral/mistral-large-3-675b-instruct) ---
Verdict: pass
Reason: The draft answer is clear, concise, and addresses the question directly. It accurately states that two-factor authentication (2FA) is not required for the Pro plan but is optional. It also correctly clarifies that 2FA is mandatory only for Enterprise accounts and provides examples of supported methods. No additional information is needed, and the response aligns with reasonable suppor

'No, two-factor authentication (2FA) is **not required** on the Pro plan—it is optional. 2FA is only mandatory for Enterprise accounts. Supported methods include authenticator apps (TOTP) and SMS.'